# 01 – Load & Clean: Greek Super League matches and betting odds (2016/17–2025/26)
**Source:** football-data.co.uk (Greece, file G1.csv per season). Column dictionary: https://www.football-data.co.uk/notes.txt

**Goal of this notebook**
1. Load the 10 season files and stack them in one table (1 row = 1 match).
2. Check which odds columns exist in each season.
3. Clean: empty rows, dates, results, missing odds, duplicates.
4. Create new columns: implied probabilities, bookmaker margin (overround), total goals.
5. Export a clean CSV and a SQLite database for the SQL and Power BI steps.

In [ ]:
import glob
import os
import sqlite3

import numpy as np
import pandas as pd

BASE = r"C:\Projects\greek-betting-markets"
RAW = os.path.join(BASE, "data-raw")
CLEAN = os.path.join(BASE, "data-clean")
os.makedirs(CLEAN, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("pandas", pd.__version__)

## 1. Load the 10 files
The season is in the file name: `G1_1617.csv` → `2016/17`. Some files have more columns than others, so we first look at each one separately.

In [ ]:
files = sorted(glob.glob(os.path.join(RAW, "G1_*.csv")))
print(len(files), "files")

raw = {}
for f in files:
    code = os.path.basename(f)[3:7]                 # '1617'
    season = f"20{code[:2]}/{code[2:]}"             # '2016/17'
    df = pd.read_csv(f, encoding="latin-1")
    df = df.dropna(how="all")                       # fully empty rows at the end of the file
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]  # empty trailing columns
    raw[season] = df
    print(f"{season}: {len(df):>4} rows, {df.shape[1]:>3} columns")

## 2. Which odds columns exist in each season?
- `B365H/D/A` = Bet365 odds for Home / Draw / Away (1X2 market)
- `PSH/D/A` = Pinnacle odds (a low-margin, "sharp" bookmaker – our benchmark)
- `B365>2.5`, `B365<2.5` = Bet365 Over / Under 2.5 goals
- `P>2.5`, `P<2.5` = Pinnacle Over / Under 2.5 goals
- columns ending in `C` (e.g. `B365CH`) = closing odds, just before kick-off

In [ ]:
check_cols = ["Time", "B365H", "PSH", "B365>2.5", "P>2.5", "Avg>2.5", "BbAv>2.5", "B365CH", "PSCH"]
availability = pd.DataFrame(
    {season: [c in df.columns for c in check_cols] for season, df in raw.items()},
    index=check_cols,
).T
availability

## 3. Stack the seasons into one table
We keep only the columns we need. `reindex` creates any missing column as empty (NaN), so all seasons get the same structure.

In [ ]:
keep = ["Div", "Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR",
        "B365H", "B365D", "B365A", "PSH", "PSD", "PSA",
        "B365>2.5", "B365<2.5", "P>2.5", "P<2.5"]

parts = []
for season, df in raw.items():
    part = df.reindex(columns=keep).copy()
    part.insert(0, "Season", season)
    parts.append(part)

matches = pd.concat(parts, ignore_index=True)
matches = matches.rename(columns={"B365>2.5": "B365_Over25", "B365<2.5": "B365_Under25",
                                  "P>2.5": "PS_Over25", "P<2.5": "PS_Under25"})
print(matches.shape)
matches.head()

## 4. Cleaning checks
### 4a. Rows without teams or result (e.g. empty lines, postponed matches)

In [ ]:
no_teams = matches["HomeTeam"].isna() | matches["AwayTeam"].isna()
no_result = matches["FTR"].isna()
print("Rows without teams :", no_teams.sum())
print("Rows without result:", no_result.sum())
matches = matches[~no_teams & ~no_result].copy()
print("Rows after removal :", len(matches))

### 4b. Dates
Older files write the year with 2 digits (`23/08/17`), newer with 4 (`23/08/2025`). `dayfirst=True` because the format is day/month/year.

In [ ]:
matches["Date"] = pd.to_datetime(matches["Date"], dayfirst=True, format="mixed")
print("Missing dates:", matches["Date"].isna().sum())
matches.groupby("Season")["Date"].agg(["min", "max", "count"])

### 4c. Results, goals and duplicates
`FTR` must be only H, D or A, and it must agree with the goals (`FTHG` home goals, `FTAG` away goals).

In [ ]:
print(matches["FTR"].value_counts())

matches["FTHG"] = matches["FTHG"].astype(int)
matches["FTAG"] = matches["FTAG"].astype(int)
expected = np.select([matches["FTHG"] > matches["FTAG"], matches["FTHG"] < matches["FTAG"]], ["H", "A"], default="D")
print("FTR that does not match the goals:", (matches["FTR"] != expected).sum())

dups = matches.duplicated(subset=["Date", "HomeTeam", "AwayTeam"]).sum()
print("Duplicate matches:", dups)

### 4d. Team names
The same club must have one spelling in all seasons, otherwise SQL and Power BI will count it as two teams.

In [ ]:
teams = pd.concat([matches["HomeTeam"], matches["AwayTeam"]]).str.strip()
team_seasons = (pd.concat([matches[["Season", "HomeTeam"]].rename(columns={"HomeTeam": "Team"}),
                           matches[["Season", "AwayTeam"]].rename(columns={"AwayTeam": "Team"})])
                .assign(Team=lambda d: d["Team"].str.strip())
                .groupby("Team")["Season"].nunique().sort_values(ascending=False))
print(len(team_seasons), "different team names")
team_seasons

### 4e. Missing odds per season
A match with missing odds stays in the table (it counts for results and home advantage), but it is left out of the margin calculations for that bookmaker.

In [ ]:
odds_cols = ["B365H", "B365D", "B365A", "PSH", "PSD", "PSA", "B365_Over25", "B365_Under25", "PS_Over25", "PS_Under25"]
for c in odds_cols:
    matches[c] = pd.to_numeric(matches[c], errors="coerce")

matches[odds_cols].isna().groupby(matches["Season"]).sum()

In [ ]:
# Odds must be above 1.00. Anything else is a data error.
bad_odds = (matches[odds_cols] <= 1).sum()
print(bad_odds[bad_odds > 0] if bad_odds.sum() else "No odds <= 1.00")
print()
matches[odds_cols].describe().T[["count", "min", "50%", "max"]]

## 5. New columns
**Implied probability** = 1 / decimal odds. Example: odds 2.00 → 50%.

**Overround (bookmaker margin)** = sum of the implied probabilities of all outcomes − 1. In a fair market the probabilities add up to 100%. The bookmaker shortens the odds, so the sum is above 100%. The extra part is the margin.

**Fair (normalised) probability** = implied probability / sum, so the three outcomes add up to exactly 100%. We use it to check if the odds are accurate.

In [ ]:
for bk in ["B365", "PS"]:
    inv = 1 / matches[[f"{bk}H", f"{bk}D", f"{bk}A"]]
    total = inv.sum(axis=1, min_count=3)                 # NaN if any of the 3 odds is missing
    matches[f"{bk}_Overround_1X2"] = total - 1
    for o in ["H", "D", "A"]:
        matches[f"{bk}_Prob{o}"] = (1 / matches[f"{bk}{o}"]) / total

    ou = (1 / matches[[f"{bk}_Over25", f"{bk}_Under25"]]).sum(axis=1, min_count=2)
    matches[f"{bk}_Overround_OU25"] = ou - 1
    matches[f"{bk}_ProbOver25"] = (1 / matches[f"{bk}_Over25"]) / ou

matches["TotalGoals"] = matches["FTHG"] + matches["FTAG"]
matches["Over25"] = (matches["TotalGoals"] > 2.5).astype(int)
matches["HomeWin"] = (matches["FTR"] == "H").astype(int)
matches["Draw"] = (matches["FTR"] == "D").astype(int)
matches["AwayWin"] = (matches["FTR"] == "A").astype(int)

# Which outcome the Bet365 odds made favourite (lowest odds)
odds3 = matches[["B365H", "B365D", "B365A"]]
has_odds = odds3.notna().all(axis=1)
matches["B365_Favourite"] = pd.Series(None, index=matches.index, dtype="object")
matches.loc[has_odds, "B365_Favourite"] = odds3[has_odds].idxmin(axis=1).str[-1]
matches["FavouriteWon"] = np.where(matches["B365_Favourite"].isna(), np.nan,
                                   (matches["B365_Favourite"] == matches["FTR"]).astype(float))

matches[["Season", "HomeTeam", "AwayTeam", "FTR", "B365H", "B365D", "B365A",
         "B365_Overround_1X2", "B365_ProbH", "PS_Overround_1X2"]].head()

## 6. First look: one row per season

In [ ]:
season_summary = matches.groupby("Season").agg(
    Matches=("FTR", "size"),
    HomeWinPct=("HomeWin", "mean"),
    DrawPct=("Draw", "mean"),
    AwayWinPct=("AwayWin", "mean"),
    AvgGoals=("TotalGoals", "mean"),
    Over25Pct=("Over25", "mean"),
    B365_Margin_1X2=("B365_Overround_1X2", "mean"),
    PS_Margin_1X2=("PS_Overround_1X2", "mean"),
    B365_Margin_OU25=("B365_Overround_OU25", "mean"),
    PS_Margin_OU25=("PS_Overround_OU25", "mean"),
)
pct_cols = [c for c in season_summary.columns if c not in ("Matches", "AvgGoals")]
season_summary.style.format({**{c: "{:.1%}" for c in pct_cols}, "AvgGoals": "{:.2f}"})

## 7. Export
- `data-clean/matches_clean.csv` → for checking and for the GitHub repo
- `data-clean/greek_betting.db` → SQLite database for DBeaver (table `matches`)

The date is saved as text `YYYY-MM-DD` so that SQLite and Power BI read it the same way.

In [ ]:
out = matches.sort_values(["Date", "HomeTeam"]).copy()
out["Date"] = out["Date"].dt.strftime("%Y-%m-%d")

csv_path = os.path.join(CLEAN, "matches_clean.csv")
db_path = os.path.join(CLEAN, "greek_betting.db")
out.to_csv(csv_path, index=False)

with sqlite3.connect(db_path) as con:
    out.to_sql("matches", con, if_exists="replace", index=False)
    n = con.execute("SELECT COUNT(*) FROM matches").fetchone()[0]

print("CSV :", csv_path, "|", len(out), "rows,", out.shape[1], "columns")
print("DB  :", db_path, "| table matches:", n, "rows")